# Benchmark for PyMieDAP

This notebook contains some tests that can tell you whether or not everything is working fine in your PyMieDAP install.

In [49]:
%matplotlib inline
import pymiedap.pymiedap as pmd
import matplotlib.pyplot as mpl  # for plotting
import numpy as np 

## Test of the Mie code
We will first check that the Mie code runs fine. For that, we use the tabulated output from **De Rooij et al. 1984**.
We define the same types of aerosols they use.

In [50]:
aerA = pmd.Aerosols(nr=[1.45], ni=[0], r_eff=0.23, v_eff=0.18)
aerB = pmd.Aerosols(nr=[1.44], ni=[0], r_eff=1.05, v_eff=0.07)
aerC = pmd.Aerosols(nr=[1.33], ni=[0], v_eff=0.07, r_eff=2, par3=0.5, psd='7')
aerD = pmd.Aerosols(nr=[1.33], ni=[0], r_eff=2.2, v_eff=0.07)

then we compute the Mie scattering

In [51]:
coefsA = pmd.mie_code(aerA, [0.55], nsubr=40, ngaur=500) # wvl= 0.55 um
coefsB = pmd.mie_code(aerB, [0.55], nsubr=40, ngaur=500) 
coefsC = pmd.mie_code(aerC, [0.70]) #wvl = 0.7 um
coefsD = pmd.mie_code(aerD, [0.70], nsubr=40, ngaur=500)

Beginning of Mie program
Wavelength 0.5500000
End of Mie program
Beginning of Mie program
Wavelength 0.5500000
End of Mie program
Beginning of Mie program
Wavelength 0.7000000
End of Mie program
Beginning of Mie program
Wavelength 0.7000000
End of Mie program


First thing we can check is that the asymmetry parameter is correct.

In [52]:
print(aerA.asym) # asymmetry parameter.
print(aerB.asym) # asymmetry parameter.
print(aerC.asym) # asymmetry parameter.
print(aerD.asym) # asymmetry parameter.

[ 0.72100229]
[ 0.7179972]
[ 0.80420101]
[ 0.80188248]


Then we can check the coefficients on some lines

In [53]:
coordsx = [0,1,2,3,0,2] #positions of alphas and betas in the coef table
coordsy = [0,1,2,3,1,3]
print(aerA.coefs[0,coordsx,coordsy,0]) #checking the values of coefficients againt de Rooij et al 1984,for l=0
print(aerA.coefs[0,coordsx,coordsy,10])  #l=10
print(aerA.coefs[0,coordsx,coordsy,15])  # l=15
print(aerA.coefs[0,coordsx,coordsy,24])  #l=24

[ 1.          0.          0.          0.92901809  0.          0.        ]
[ 0.08802913  0.11121631  0.10151652  0.0832367  -0.00358954  0.02614555]
[ 0.00228455  0.00218155  0.00224082  0.00255047  0.00033707  0.00046371]
[  5.48015333e-06   6.77440344e-06   2.37713944e-06   1.97521301e-06
  -7.37991466e-07   3.53992457e-06]


And for aerosols B, C and D.

In [54]:
print(aerB.coefs[0,coordsx,coordsy,0])
print(aerB.coefs[0,coordsx,coordsy,10])
print(aerB.coefs[0,coordsx,coordsy,15])
print(aerB.coefs[0,coordsx,coordsy,24])

[ 1.          0.          0.          0.86462372  0.          0.        ]
[ 5.06996479  5.33408747  5.31811435  5.12294652  0.03998552  0.02898462]
[ 4.29325014  4.22823484  4.16063179  4.35190093 -0.04425792  0.31830627]
[ 1.48138437  1.66704814  1.50932983  1.42450347 -0.04210442  0.24849267]


In [55]:
print(aerC.coefs[0,coordsx,coordsy,0])
print(aerC.coefs[0,coordsx,coordsy,10])
print(aerC.coefs[0,coordsx,coordsy,15])
print(aerC.coefs[0,coordsx,coordsy,80])

[ 1.          0.          0.          0.95461472  0.          0.        ]
[ 1.17713561  1.28475848  1.26595458  1.17411334 -0.02230815  0.09498393]
[ 0.42561196  0.42569828  0.42313876  0.42980116 -0.00553692  0.04263078]
[  1.28685332e-06   1.40261583e-06   1.05753279e-06   1.01412365e-06
   8.22672010e-08   4.74615747e-07]


In [56]:
print(aerD.coefs[0,coordsx,coordsy,0])
print(aerD.coefs[0,coordsx,coordsy,10])
print(aerD.coefs[0,coordsx,coordsy,15])
print(aerD.coefs[0,coordsx,coordsy,80])

[ 1.          0.          0.          0.91852089  0.          0.        ]
[ 7.20704159  7.35295547  7.25693617  7.13468861 -0.029961    0.09669141]
[ 7.98539793  8.00874554  7.98884822  7.99970188  0.02916147  0.16475289]
[  3.04667428e-03   3.18324854e-03   2.83399090e-03   2.77620381e-03
   2.74463843e-05   5.59438200e-04]


## Test of the DAP code

in their paper, de Haan et al, have tested two atmosphere models:
* one with an homogeneous atmosphere above a black surface, only with aerosols and opacity=1 (we'll call it modelB)
* one with a Lambertian surface (A=0.1) and two layers: (modelA)
    * an upper layer of molecules with opacity 0.1
    * a lower layer with gas and aerosols mixed such as the molecular opt. thickness is 0.1 and aerosol opt. thickness is 0.4
    
Molecular depolarization is 0.0279. The aerosols are those of the type C defined above.

In [67]:
modelA = pmd.Model()
modelA.wvl_list = [0.7]
del modelA.layers.gasbelow
del modelA.layers.haze
modelA.layers.gastop.rayscat=False
modelA.layers.gastop.tau_ray=[0.1]
modelA.layers.cloud.rayscat=False
modelA.layers.cloud.tau = [0.4]
modelA.layers.cloud.tau_ray = [0.1]
modelA.layers.cloud.aerosols = aerC
modelA.dpol=0.0279
modelA.surface[0,0] = 0.1

modelB = pmd.Model()
modelB.wvl_list = [0.7]
del modelB.layers.gastop
del modelB.layers.haze
del modelB.layers.cloud
modelB.layers.gasbelow.rayscat=False
modelB.layers.gasbelow.tau=[1.0]
modelB.layers.gasbelow.aerosols = aerC
modelB.surface[0,0] = 0.0


In [68]:
modelA
modelB.layers.gasbelow.aerosols

Spherical particles
nr =[1.33]))
ni =[0]))
   R_eff = 2.000 
 Type: C

In [69]:
mus = np.array([0.1])#, 0.5, 1.0, 0.1, 0.5, 1.0])
emissions = np.degrees(np.arccos(mus))
mu0s = np.array([0.5])#, 0.5, 0.5, 0.1, 0.1, 0.1])
szas = np.degrees(np.arccos(mu0s))
dphi = np.radians([0])#, 0, 0, 30, 30, 30])
alphas = mus*mu0s  + np.sqrt(1-mus**2)*np.sqrt(1-mu0s**2)*np.cos(dphi) 
phases = np.degrees(np.arccos(alphas))
print(phases)
print(emissions)
print(szas)
pmd.calc_azimuth(phases,szas,emissions, deg=True)

[ 24.26082952]
[ 84.26082952]
[ 60.]


array([  8.53773646e-07])

In [70]:
pmd.compute_model(modelA, output_name='modelA', rename=True, nmat=4, nmug=22)
pmd.compute_model(modelB, output_name='modelB', rename=True, nmat=4)
print(modelA.name)

In layer gastop:
Beginning of Mie program
Wavelength 0.7000000
End of Mie program
Aerosols mixed!
In layer cloud:
Beginning of Mie program
Wavelength 0.7000000
End of Mie program
Aerosols mixed!
Beginning of DAP program
Wavelength 0.7000000 microns
C.sc.0.7000000
C.sc.0.7000000
fou_0.7000000.dat
End of DAP program
In layer gasbelow:
Beginning of Mie program
Wavelength 0.7000000
End of Mie program
Aerosols mixed!
Beginning of DAP program
Wavelength 0.7000000 microns
C.sc.0.7000000
fou_0.7000000.dat
End of DAP program
['dap_database/modelA_0.7000000.dat']


In [71]:
Ia,Qa,Ua,Va = pmd.read_dap_output(phases, szas, emissions, modelA.name[0])
Ib,Qb,Ub,Vb = pmd.read_dap_output(phases, szas, emissions, modelB.name[0])
#Ia,Qa,Ua,Va = pmd.read_dap_output(phases, szas[:,np.newaxis], emissions, modelA.name)

In [73]:
Ia,Qa,Ua,Va

(array([ 0.23610045]),
 array([-0.00515991]),
 array([ -7.51119756e-10]),
 array([ 0.]))

In [74]:
Ib,Qb,Ub,Vb

(array([ 0.06357253]),
 array([-0.0039328]),
 array([ -6.55807352e-11]),
 array([ 0.]))